# Multi-Camera ANPR — Vehicle Detection, Tracking & Plate Crop Selection

**SIH 2026 · Problem Statement 26127 · Bharat Electronics Limited**
*City-Wide AI Engine for Multi-Camera ANPR Trajectory Tracking and Urban Traffic Analytics*

---

This notebook covers **one stage** of the pipeline: turning camera frames into
ranked license-plate crops, one record per vehicle.

It does **not** read plates. Recognition is a separate stage.

| Stage | Owner | Status |
|---|---|---|
| 1. Video ingest (RTSP, buffering) | A | in progress |
| **2. Vehicle + plate detection** | **B — this notebook** | **done** |
| **3. Multi-object tracking** | **B — this notebook** | **done** |
| **4. Best-frame selection** | **B — this notebook** | **done** |
| 5. OCR + per-character confidence | C | in progress |
| 6. Cross-camera identity resolution | backend | not started |

### What this stage outputs

```
{
  global_id      : "CAM_MG_ROAD_01-1788006071-000007",
  vehicle_class  : "car",
  plate_status   : "PENDING_OCR" | "UNREAD" | "NO_PLATE",
  crops          : [5 images, ranked best-first],
  crop_quality   : [0.79, 0.79, 0.78, 0.77, 0.76],
  direction_bearing_deg : 104.4,
  track_frames   : 57
}
```

**Five crops, not one.** Section 6 shows the measurement that justifies this.

## 1. Setup

Runs on Colab (free CPU tier) or locally. `lap` is required by the tracker and
Ultralytics will otherwise try to download it at first use — which fails on an
offline machine, so we install it explicitly.

In [ ]:
!pip install -q ultralytics lap huggingface_hub
import ultralytics, torch, cv2, numpy as np
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__)
print("opencv     ", cv2.__version__)

### Models

Two pretrained detectors. **Neither was trained by us**, and both are disclosed
on the references slide.

| Model | Source | Licence | Trained by us? |
|---|---|---|---|
| `yolo11n.pt` — vehicles | Ultralytics, COCO | AGPL-3.0 | No — `car`/`bus`/`truck`/`motorcycle` are COCO classes |
| `license-plate-finetune-v1m.pt` — plates | `morsetechlab` on HuggingFace | AGPL-3.0 | No — 300 epochs on an A100 by the author |

The plate model's own card reports mAP@50 of 0.9813 **but states the upstream
Roboflow dataset has train/test contamination and the metrics are inflated.**
We therefore ignore that figure entirely and measure our own (section 5).

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

PLATE_W = hf_hub_download("morsetechlab/yolov11-license-plate-detection",
                          "license-plate-finetune-v1m.pt")
vehicle_model = YOLO("yolo11n.pt")
plate_model   = YOLO(PLATE_W)

print("vehicle classes used:",
      {k: v for k, v in vehicle_model.names.items()
       if v in ("car", "bus", "truck", "motorcycle")})

## 2. Why two detection passes

A plate ~30 px wide in a 1920 px frame becomes ~10 px once YOLO downscales to
640. There is nothing left to detect.

So we detect the **vehicle** in the full frame, crop it, and run plate detection
**inside that crop** — where the same plate is ~150 px of a 400 px image.

```
full frame  ──▶ vehicle detector ──▶ vehicle boxes ──┐
                                                     ▼
                                  crop ──▶ plate detector ──▶ plate box
                                                     │
                            (coords are relative to the crop —
                             add the crop offset exactly once)
```

The cost is one plate inference per vehicle per frame, which dominates runtime.
Section 7 covers how that is reduced by ~90% without losing crops.

## 3. Why tracking is not optional

Without it, a car visible for 2 seconds at 10 fps produces **20 separate
records**. Three consequences:

1. The backend receives 20 rows for one vehicle and has to invent deduplication
2. Traffic counts inflate by a factor that depends on vehicle speed — a car
   stopped at a signal counts hundreds of times
3. You discard 20 looks at the same plate and use one

ByteTrack (built into Ultralytics) assigns a stable ID across frames. We emit
**one event when the vehicle leaves**, carrying its best crops.

> **`persist=True` is the entire mechanism.** Without it every call restarts
> numbering from 1 and there is no tracking at all.

### Track IDs are not unique over time

ByteTrack recycles an ID once a vehicle leaves, so two different cars 30 seconds
apart are both "track 20" — observed in our own runs. Scoping by session was
still not enough, because reuse happens *within* a single run. The ID is
therefore an internal monotonic counter, never ByteTrack's number:

```
CAM_MG_ROAD_01 - 1788006071 - 000007
    camera         session      counter
```

## 4. The pipeline

`detector.py` holds the implementation. Upload it, or clone the repo.

In [ ]:
# Colab: upload detector.py
# from google.colab import files; files.upload()

import detector
from detector import PlateDetector, VEHICLE_CLASSES

print("thresholds in use\n")
for k in ["MIN_PLATE_WIDTH_PX", "MIN_CROP_WIDTH_PX", "PLATE_MIN_BOX_W",
          "PLATE_ASPECT_MIN", "PLATE_ASPECT_MAX", "PLATE_MAX_WIDTH_FRAC",
          "PLATE_MAX_AREA_FRAC", "PLATE_PAD_X_FRAC", "PLATE_PAD_Y_FRAC",
          "MIN_TRACK_FRAMES", "MIN_VEHICLE_WIDTH_FOR_PLATE",
          "PLATE_DETECT_EVERY_N", "TOP_K_CROPS"]:
    print(f"  {k:32} = {getattr(detector, k)}")

### The geometric filters, and why these numbers

Off-the-shelf plate detectors fire on bus destination boards, advertising
panels and painted truck signage. In one 648-frame clip we saw a "plate"
**547 px wide** — it was the side of a bus.

Rather than trust the detector's confidence, we check whether the box could
physically be a plate. Indian plate standards give the ratios directly:

| Standard | Dimensions | Aspect |
|---|---|---|
| Single-row | 500 × 120 mm | ≈ 4.2 |
| Two-row (motorcycles) | 285 × 200 mm | ≈ 1.4 |

So the accepted band is 1.1–7.0, generous enough for perspective
foreshortening. Plus two relative checks: a plate cannot exceed 45% of its
vehicle's width or 15% of its area.

**One subtlety that matters:** candidates are filtered *first*, then the most
confident survivor is chosen. Picking max-confidence before filtering lets a
large confident bus panel beat a small correct plate on the same vehicle.

In [ ]:
import matplotlib.pyplot as plt

VIDEO = "sample.mp4"   # <-- your clip

cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_FRAMES, 120)
ok, frame = cap.read(); cap.release()
assert ok, f"could not read {VIDEO}"

res = vehicle_model.predict(frame, classes=list(VEHICLE_CLASSES),
                            conf=0.35, imgsz=960, verbose=False)[0]

vis = frame.copy()
for box, cls in zip(res.boxes.xyxy.cpu().numpy().astype(int),
                    res.boxes.cls.cpu().numpy().astype(int)):
    x1, y1, x2, y2 = box
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 255), 2)
    cv2.putText(vis, VEHICLE_CLASSES.get(cls, "?"), (x1, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)

    pr = plate_model.predict(frame[y1:y2, x1:x2], conf=0.25, verbose=False)[0]
    for pb in pr.boxes.xyxy.cpu().numpy().astype(int):
        cv2.rectangle(vis, (pb[0] + x1, pb[1] + y1),
                      (pb[2] + x1, pb[3] + y1), (0, 255, 0), 2)

plt.figure(figsize=(15, 9))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title("orange = vehicle (pass 1)   green = plate (pass 2, inside crop)")
plt.show()

### Full run — one event per vehicle

In [ ]:
# {PLATE_W} interpolates the PYTHON variable into the shell magic.
# $PLATE_W would look for a shell env var and silently pass nothing.
!python detector.py --source sample.mp4 --stride 3 --camera-id CAM_MG_ROAD_01 --plate-weights "{PLATE_W}" --save-crops out/

### The crops that go to the OCR stage

In [ ]:
import os, glob

files = sorted(glob.glob("out/*.jpg"))[:15]
if files:
    fig, axes = plt.subplots((len(files) + 2) // 3, 3, figsize=(14, 1.6 * len(files) / 3 + 3))
    for ax, f in zip(axes.ravel(), files):
        ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB),
                  interpolation="nearest")   # no smoothing: judge real pixels
        ax.set_title(os.path.basename(f).split("-")[-1], fontsize=7)
        ax.axis("off")
    for ax in axes.ravel()[len(files):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
    print(f"{len(glob.glob('out/*.jpg'))} crops total")
else:
    print("no crops — check --plate-weights was passed")

## 5. Measured results

Every crop was judged by hand — *plate* (all characters legible), *partial*
(a real plate, not fully legible), *not a plate*. Four sources, two countries,
two resolutions.

| Source | Resolution | Events | Fragments | Vehicles w/ crops | Vehicles w/ real plate |
|---|---|---|---|---|---|
| Kolkata street (India) | 1080p | 37 | 21% | 14 | **12 (86%)** |
| UA-DETRAC MVI_39031 (China) | 960×540 | 80 | 25% | 5 | 3 (60%) |
| UA-DETRAC MVI_39371 (China) | 960×540 | 83 | 24% | 13 | 6 (46%) |
| UA-DETRAC MVI_39311 (China) | 960×540 | 52 | 8% | 0 | 0 |

### Finding 1 — tracking generalises

Fragment rate held at **21–25%** across two countries and four camera positions,
with individual tracks surviving 100–350 consecutive frames. The tracker was
never tuned per-clip.

*(MVI_39311 is 8% because traffic was stationary — a dozen vehicles were tracked
for the entire clip without the track breaking. Correct behaviour, not an
outlier.)*

### Finding 2 — plate yield is resolution-bound, not model-bound

86% at 1080p, 46–60% at 960×540, 0% where vehicles never come close.

This is a **camera-placement result, not an algorithm result**. Government ANPR
installations use narrow-FOV cameras aimed down a single lane so plates land at
a consistent 100–200 px. A wide scene camera watching a whole junction cannot
support plate reading at any resolution, and no model fixes that.

In [ ]:
import matplotlib.pyplot as plt

src   = ["Kolkata\n1080p", "39031\n960p", "39371\n960p", "39311\n960p"]
yield_= [86, 60, 46, 0]
frag  = [21, 25, 24, 8]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar(src, yield_, color=["#2a7", "#69c", "#69c", "#69c"])
ax[0].set_title("Vehicles yielding a real plate (%)"); ax[0].set_ylim(0, 100)
ax[1].bar(src, frag, color="#c86")
ax[1].set_title("Track fragment rate (%)"); ax[1].set_ylim(0, 40)
for a in ax: a.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## 6. The result that justifies sending five crops

For each vehicle we compared two policies against the hand-labelled ground
truth:

* **rank-0 only** — send the single highest-quality crop (what most pipelines do)
* **all crops** — send up to five and let the recognition stage vote

| Source | Vehicles | rank-0 only | all crops | recovered |
|---|---|---|---|---|
| Kolkata | 14 | 7 | **9** | +2 |
| MVI_39031 | 5 | 2 | **3** | +1 |
| MVI_39371 | 13 | 3 | **4** | +1 |
| **Total** | **32** | **12** | **16** | **+4** |

**16 vehicles instead of 12 — a 33% relative improvement, at zero model cost.**
It held in all three audited datasets independently.

### A concrete case

MVI_39371, vehicle 000029:

| crop | quality | verdict |
|---|---|---|
| `_0` | **0.75** — highest in the entire run | not a plate |
| `_1` | 0.55 | **plate** |
| `_2` | 0.53 | **plate** |
| `_3` | 0.52 | **plate** |
| `_4` | 0.48 | **plate** |

A single-best-frame pipeline gets this vehicle wrong. Ours gets it right four
times over.

### An honest negative result about our own heuristic

The quality score ranks crops by sharpness, pixel width and detector
confidence. Measured against ground truth, **it carries essentially no signal
about whether a crop is actually a plate** — real plates and false positives
overlap almost completely in score.

The reason is straightforward: a flat printed sticker or a bus route panel is
genuinely *sharper and larger* than a slightly angled, slightly dirty plate. The
function measures what it was asked to measure; plate-ness is not a geometric
property.

We are not fixing this by re-weighting, because the missing signal is textual.
`NOREFUSAL` (a Kolkata taxi sticker) and `TATA` (painted truck signage) both
fail Indian RTO plate grammar and are rejected downstream at no cost. Geometry
filters shape; grammar filters content. **Neither alone is sufficient** — and
this is exactly why the five-crop output matters.

## 7. Compute budget

Plate detection runs per vehicle per frame and accounted for ~94% of runtime.
Nearly all of it was waste, for two reasons:

1. **A vehicle narrower than 120 px cannot contain a readable plate.** Plates
   are ~¼ of vehicle width, so a 120 px car gives a ~30 px plate, which the box
   floor rejects anyway. Skip it before paying for it.
2. **Only 5 crops per vehicle are kept**, so running detection on all 300 frames
   of a long track is pointless.

We sample every 4th frame per track — **plus always when the vehicle box is the
largest it has been**, since that is its closest approach and its best plate.
Naive sampling could miss that frame.

Measured: **~90% fewer plate inferences, identical output.**

For live deployment this matters more, not less: a camera delivers 25–30 fps and
only 5–8 need processing. A vehicle is in frame 2–3 seconds; ninety looks at it
are not required.

In [ ]:
import time
det = PlateDetector("yolo11n.pt", PLATE_W, camera_id="BENCH")
cap = cv2.VideoCapture(VIDEO)
frames = []
for _ in range(60):
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
cap.release()

det.process_frame(frames[0])                      # warm-up, excluded
t0 = time.time()
for f in frames[1:]:
    det.process_frame(f)
el = time.time() - t0

print(f"{len(frames)-1} frames in {el:.1f}s  ->  {(len(frames)-1)/el:.1f} fps")
print(f"plate inferences: {det.n_plate_infer} run, {det.n_plate_skipped} skipped")
if det.n_plate_infer + det.n_plate_skipped:
    saved = 100*det.n_plate_skipped/(det.n_plate_infer+det.n_plate_skipped)
    print(f"saved: {saved:.0f}%")
print("\nThis fps figure is the input to per-node camera capacity.")

## 8. Limitations

Stated plainly, because each one has a measurement behind it.

1. **Validated mainly on non-Indian footage.** One Indian clip (Kolkata, 1080p);
   the rest is Chinese traffic surveillance. No public dataset of Indian
   government ANPR video with readable plates exists — the one Indian government
   CCTV dataset we found (BMD-45, Bengaluru Traffic Police) has license plates
   deliberately blurred for privacy. This is a real gap and we do not paper over it.

2. **Two-wheelers untested.** The plate model's authors state performance on
   motorcycles is not guaranteed. Our single motorcycle appeared at 22 px —
   too small to conclude anything. Given the share of two-wheelers in Indian
   traffic, this is the most important open question in this stage.

3. **The filters are unmeasured in one direction.** We know they reject 139–284
   boxes per clip. We do not know how many were real plates, because rejected
   boxes are not saved. Recall against ground truth requires a labelled dataset;
   RodoSol-ALPR has been requested for exactly this.

4. **Quality ranking carries no plate-ness signal** — see section 6.

5. **Not production-grade.** No RTSP reconnection, no error recovery, single
   stream, thresholds tuned on limited data.

## 9. Next

| | |
|---|---|
| RodoSol-ALPR (toll-booth geometry, motorcycles, plate labels) | requested |
| UFPR-ALPR (30 consecutive frames/vehicle — validates temporal voting) | requested |
| Indian government CCTV via institutional request | drafting |
| Fine-tune the plate detector on Indian footage | pending recall measurement |

## References

- Ultralytics YOLO11 / ByteTrack — AGPL-3.0
- morsetechlab, *YOLOv11 License Plate Detection*, HuggingFace — AGPL-3.0
- Wen et al., *UA-DETRAC: A New Benchmark and Protocol for Multi-Object Detection and Tracking*
- Laroca et al., *On the Cross-dataset Generalization in License Plate Recognition*, VISAPP 2022
- Laroca et al., *A Robust Real-Time Automatic License Plate Recognition Based on the YOLO Detector*, IJCNN 2018